# Extracting Categories with `cat.extract()`

`cat.extract()` discovers and returns a **clean, deduplicated set of categories** from your text data. It runs the same chunking and iteration process as `explore()`, then normalizes, deduplicates, and semantically merges the results.

Use `extract()` when you want a ready-to-use category list — typically as a first step before `classify()`.

**What this notebook covers:**
- Basic category extraction
- Understanding the output (counts DataFrame, top categories)
- Controlling the number and granularity of categories
- Steering extraction with `focus` and `research_question`
- Feeding discovered categories into `classify()`
- Text, image, and PDF extraction
- Switching providers and using local models

**What you need:**
- Python 3.9+
- `pip install catllm` (or `pip install catllm[pdf]` for PDF support)
- An API key from at least one supported provider

**See also:** `Exploring Categories with explore().ipynb` for raw frequency analysis and saturation curves.

---

## How `extract()` Works

1. **Divide** your data into `divisions` chunks (default 12)
2. **Send** each chunk to the LLM, asking it to identify categories
3. **Repeat** for `iterations` passes (default 8), reshuffling data each time
4. **Normalize** category names, **deduplicate** similar ones, and **merge** semantically equivalent categories
5. **Return** the top `max_categories` with frequency counts

In [ ]:
import catllm as cat
import pandas as pd

## Setup

In [ ]:
# Replace with your actual API key
api_key = "YOUR_API_KEY"

## Sample Data

In [ ]:
responses = [
    "because i dont like living here",
    "for a bigger house",
    "to be with my wife",
    "got a new job in another state",
    "rent was too expensive so we had to leave",
    "my landlord sold the building so I had to find a new place",
    "wanted to be closer to my kids' school",
    "the neighborhood was getting unsafe",
    "retired and wanted to move somewhere warmer",
    "my partner got transferred to a different city",
    "needed more space after the baby was born",
    "commute was too long from where we were living",
    "moved in with my boyfriend",
    "lost my job and couldn't afford the mortgage",
    "just wanted a change of scenery honestly",
    "parents are getting older and need help",
    "flooding damaged our house and we had to relocate",
    "got accepted to graduate school out of state",
    "divorce, had to sell the house",
    "the apartment complex was being torn down for redevelopment",
]

## Basic Extraction

The simplest usage requires `input_data`, a `survey_question` for context, and an `api_key`. The output is a dict with:

- **`top_categories`** — List of the final category names (ready to pass to `classify()`)
- **`counts_df`** — DataFrame showing each category and how often it was discovered
- **`raw_top_text`** — Raw model output from the final merge step

In [ ]:
results = cat.extract(
    input_data=responses,
    survey_question="Why did you move to your current residence?",
    api_key=api_key,
    user_model="gpt-4o-mini",
    iterations=5,
    divisions=4,
)

print("Top categories:")
for i, cat_name in enumerate(results["top_categories"], 1):
    print(f"  {i}. {cat_name}")

## Understanding the Counts DataFrame

The `counts_df` shows how often each category was discovered across all iterations. Higher counts indicate more robust categories that the model consistently identifies.

In [ ]:
results["counts_df"]

## Controlling the Number of Categories

Use `max_categories` to control how many final categories are returned. The model discovers as many as it can, then the top N by frequency are kept.

In [ ]:
# Extract just 5 categories
results_5 = cat.extract(
    input_data=responses,
    survey_question="Why did you move to your current residence?",
    api_key=api_key,
    user_model="gpt-4o-mini",
    max_categories=5,
    iterations=5,
    divisions=4,
)

print(f"Got {len(results_5['top_categories'])} categories:")
for i, cat_name in enumerate(results_5["top_categories"], 1):
    print(f"  {i}. {cat_name}")

## Controlling Granularity: `specificity`

- **`"broad"`** (default): High-level themes (e.g., "Financial reasons")
- **`"specific"`**: Granular categories (e.g., "Rent increase", "Mortgage affordability")

In [ ]:
results_specific = cat.extract(
    input_data=responses,
    survey_question="Why did you move to your current residence?",
    api_key=api_key,
    user_model="gpt-4o-mini",
    specificity="specific",
    max_categories=10,
    iterations=5,
    divisions=4,
)

print("Specific categories:")
for i, cat_name in enumerate(results_specific["top_categories"], 1):
    print(f"  {i}. {cat_name}")

## Steering with `focus`

The `focus` parameter tells the model what aspects to emphasize. Useful when your data covers many topics but you only care about a subset.

In [ ]:
results_focused = cat.extract(
    input_data=responses,
    survey_question="Why did you move to your current residence?",
    focus="family and relationship motivations",
    api_key=api_key,
    user_model="gpt-4o-mini",
    max_categories=8,
    iterations=5,
    divisions=4,
)

print("Focused on family/relationships:")
for i, cat_name in enumerate(results_focused["top_categories"], 1):
    print(f"  {i}. {cat_name}")

## Research Context

`research_question` provides broader academic context. While `survey_question` tells the model *what* the data is, `research_question` tells it *why* you're analyzing it.

In [ ]:
results_research = cat.extract(
    input_data=responses,
    survey_question="Why did you move to your current residence?",
    research_question="What push and pull factors drive residential mobility among US adults?",
    api_key=api_key,
    user_model="gpt-4o-mini",
    max_categories=8,
    iterations=5,
    divisions=4,
)

print("With research context:")
for i, cat_name in enumerate(results_research["top_categories"], 1):
    print(f"  {i}. {cat_name}")

## Using Extracted Categories for Classification

The typical workflow: extract categories first, then pass them to `classify()`. This gives you more control than `categories="auto"` in `classify()`, since you can inspect and adjust the categories before classifying.

In [ ]:
# Step 1: Extract
results = cat.extract(
    input_data=responses,
    survey_question="Why did you move to your current residence?",
    api_key=api_key,
    user_model="gpt-4o-mini",
    max_categories=8,
    iterations=5,
    divisions=4,
)

print("Discovered categories:")
for i, cat_name in enumerate(results["top_categories"], 1):
    print(f"  {i}. {cat_name}")

# Step 2: Classify using discovered categories
classified = cat.classify(
    input_data=responses,
    categories=results["top_categories"],
    survey_question="Why did you move to your current residence?",
    api_key=api_key,
    user_model="gpt-4o-mini",
)

classified.head()

## Saving Results

Pass a `filename` to save the extracted categories (with counts) to a CSV file.

In [ ]:
results_saved = cat.extract(
    input_data=responses,
    survey_question="Why did you move to your current residence?",
    api_key=api_key,
    user_model="gpt-4o-mini",
    max_categories=8,
    iterations=5,
    divisions=4,
    filename="extracted_categories.csv",
)

saved_df = pd.read_csv("extracted_categories.csv")
print(f"Saved to extracted_categories.csv")
saved_df

## Switching Providers

CatLLM auto-detects the provider from the model name.

In [ ]:
# Anthropic
results_anthropic = cat.extract(
    input_data=responses,
    survey_question="Why did you move to your current residence?",
    api_key="YOUR_ANTHROPIC_KEY",
    user_model="claude-sonnet-4-20250514",
    max_categories=8,
    iterations=5,
    divisions=4,
)

print("Anthropic categories:")
for i, cat_name in enumerate(results_anthropic["top_categories"], 1):
    print(f"  {i}. {cat_name}")

## Local Models with Ollama

Run extraction entirely locally using [Ollama](https://ollama.com/download). No API key needed.

Requires Ollama installed and running (`ollama serve`).

In [ ]:
results_local = cat.extract(
    input_data=responses,
    survey_question="Why did you move to your current residence?",
    api_key="not-needed",
    user_model="llama3.2",
    model_source="ollama",
    auto_download=True,
    max_categories=8,
    iterations=5,
    divisions=4,
)

print("Ollama categories:")
for i, cat_name in enumerate(results_local["top_categories"], 1):
    print(f"  {i}. {cat_name}")

## Image and PDF Extraction

`extract()` also works with images and PDFs. Set `input_type` to `"image"` or `"pdf"` and point `input_data` to a directory, single file, or list of file paths.

PDF support requires: `pip install catllm[pdf]`

In [ ]:
# Extract categories from images
# results_img = cat.extract(
#     input_data="/path/to/images/",
#     input_type="image",
#     survey_question="Product photos",
#     api_key=api_key,
#     user_model="gpt-4o",
# )

# Extract categories from PDFs
# results_pdf = cat.extract(
#     input_data="/path/to/pdfs/",
#     input_type="pdf",
#     survey_question="Research papers on climate change",
#     mode="text",
#     api_key=api_key,
#     user_model="gpt-4o",
# )

## Parameter Reference

| Parameter | Default | Effect |
|-----------|---------|--------|
| `survey_question` | `""` | The survey question or description of the data. |
| `input_type` | `"text"` | `"text"`, `"image"`, or `"pdf"`. |
| `max_categories` | `12` | Maximum number of final categories after merging. |
| `categories_per_chunk` | `10` | Categories to extract per chunk. |
| `divisions` | `12` | Chunks per iteration. |
| `iterations` | `8` | Number of passes over the data. |
| `specificity` | `"broad"` | `"broad"` for high-level themes, `"specific"` for granular categories. |
| `focus` | `None` | Steer extraction toward a topic. |
| `research_question` | `None` | Academic context to guide extraction. |
| `creativity` | `None` | Temperature (0.0–1.0). |
| `mode` | `"text"` | PDF processing mode: `"text"`, `"image"`, or `"both"`. |
| `random_state` | `None` | Seed for reproducible chunking. |
| `chunk_delay` | `0.0` | Seconds between API calls. |
| `filename` | `None` | Save results to a CSV file. |

**Typical workflow:**
```python
# 1. Extract clean categories
results = cat.extract(input_data=texts, survey_question="...", api_key=key)

# 2. Review
print(results['top_categories'])

# 3. Classify
classified = cat.classify(input_data=texts, categories=results['top_categories'], api_key=key)
```

**Supported providers:** OpenAI, Anthropic, Google, Mistral, Perplexity, xAI, HuggingFace, Ollama (local).